In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/regression220718/sample_submission.csv
/kaggle/input/regression220718/train.csv
/kaggle/input/regression220718/test.csv


# noise list
- POSTCODE
- LATITUDE
- LONGITUDE
- NEAREST_STN
- NEAREST_SCH
- ID
- ADDRESS

# discrete list

- SUBURB

# Binary
- BUILD_YEAR_NAN (NEW)

# continous list

- BEDROOMS
- BATHROOMS
- GARAGE
- FLOOR_AREA
- LAND_AREA 
- BUILD_YEAR #현재 년도에서 뺀값으로 변경 필요
- CBD_DIST
- NEAREST_STN_DIST
- DATE_SOLD #이거도 현재 년도에서 뺀값으로 변경 필요
- NEAREST_SCH_DIST
- NEAREST_SCH_RANK

# target

- PRICE

In [2]:
#Data Load
data = pd.read_csv('../input/regression220718/train.csv')
target = data.PRICE
data.drop(['ID','NEAREST_SCH','NEAREST_STN','LONGITUDE','LATITUDE','POSTCODE','PRICE','ADDRESS'], axis=1, inplace=True)
#max(data.BUILD_YEAR)#2017
#max(data.DATE_SOLD)#12-2020 ->2021-01을 기준으로 만들자
#data.DATE_SOLD.isna().sum()#0
#data.ADDRESS.isna().sum()#0
#data.SUBURB.isna().sum()#0
index = np.random.permutation(len(data))
train_size = int(len(data)*0.8)
ti = index[:train_size]
vi = index[train_size:]
data_train = data.iloc[ti]
data_valid = data.iloc[vi]
target_train = target.iloc[ti]
target_valid = target.iloc[vi]
#data.BEDROOMS.isna().sum()0
#data.BATHROOMS.isna().sum()0
#data.GARAGE.isna().sum()1367
#data.FLOOR_AREA.isna().sum()0
#data.LAND_AREA.isna().sum()0
#data.CBD_DIST.isna().sum()0
#data.NEAREST_STN_DIST.isna().sum()0
#data.NEAREST_SCH_DIST.isna().sum()0
#data.NEAREST_SCH_RANK.isna().sum()6000

In [3]:
def transfer_date_sold(x):
    t_str = x[:-1].split('-')
    return (2020-int(t_str[1]))*12+13-int(t_str[0])

train_cat={}#트레인 카테고리 기억용 전역변수
def get_onehot(data, col_label, fit):
    src_col = data.loc[:,col_label]
    cat_1 = None
    if fit:
        cat_1= sorted(list(set(src_col))) #카테고리를 이름순으로 나열
        cat_1.append("nan@!")
        train_cat[col_label] = cat_1
    else:
        cat_1 = train_cat[col_label]
    one_1 = np.zeros((len(src_col), len(cat_1))) #새 컬럼 만들음.
    
    for idx in range(len(src_col)):
        t_cat = src_col.loc[src_col.index[idx]]
        if t_cat in cat_1:
            cat_idx = cat_1.index(t_cat)
        else:
            cat_idx = len(cat_1)-1
        one_1[idx][cat_idx] = 1
    
    dst_col_labels = []
    for cat in cat_1:
        dst_col_labels.append(col_label+'_'+cat)
        
    return pd.DataFrame(one_1, columns=dst_col_labels, index=src_col.index)

train_vars={}

def preprocess(data, fit=False):
    result_data = data.copy()
    #데이터 산술적 변환, 및 생성
    result_data.loc[result_data['BUILD_YEAR'].isna(),'BUILD_YEAR'] = 2020#dataframe에 속한 데이터를 넣을때는 꼭 dataframe에 대한 직접참조하기.
    result_data['BUILD_YEAR'] = result_data['BUILD_YEAR'].astype(np.int32)
    result_data['BUILD_YEAR'] = 2020-result_data['BUILD_YEAR']
    result_data['BUILD_YEAR_NAN'] = (result_data['BUILD_YEAR'] == 0).astype(np.int32)
    
    result_data['DATE_SOLD'] = result_data["DATE_SOLD"].apply(transfer_date_sold)
    
    if fit:
        train_vars['GARAGE_AVG'] = result_data['GARAGE'].median()
    result_data['GARAGE_NAN'] = result_data['GARAGE'].isna().astype(np.int32)
    result_data.loc[result_data['GARAGE'].isna(), 'GARAGE'] = train_vars['GARAGE_AVG']
    if fit:
        train_vars['NEAREST_SCH_RANK_AVG'] = result_data['NEAREST_SCH_RANK'].median()
    result_data['NEAREST_SCH_RANK_NAN'] = result_data['NEAREST_SCH_RANK'].isna().astype(np.int32)
    result_data.loc[result_data['NEAREST_SCH_RANK'].isna(),'NEAREST_SCH_RANK'] = train_vars['NEAREST_SCH_RANK_AVG']
    #one-hot
    
    return pd.concat([get_onehot(result_data,'SUBURB', fit), result_data.drop(['SUBURB'],axis=1)], axis=1)

In [4]:
preprocess(data, True)

,SUBURB_Alexander Heights,SUBURB_Alfred Cove,SUBURB_Alkimos,SUBURB_Anketell,SUBURB_Applecross,SUBURB_Ardross,SUBURB_Armadale,SUBURB_Ascot,SUBURB_Ashby,SUBURB_Ashfield,...,FLOOR_AREA,BUILD_YEAR,CBD_DIST,NEAREST_STN_DIST,DATE_SOLD,NEAREST_SCH_DIST,NEAREST_SCH_RANK,BUILD_YEAR_NAN,GARAGE_NAN,NEAREST_SCH_RANK_NAN
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,266,13,35500,3000,64,1.079230,120.0,0,0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,301,6,16900,8100,30,0.667585,68.0,0,0,1
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,220,28,33500,3500,11,0.650761,111.0,0,0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,185,82,3500,1500,6,0.484465,106.0,0,0,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,186,16,12900,800,39,1.869050,25.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18505,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,198,13,21200,4400,33,1.104533,131.0,0,0,0
18506,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,237,10,22100,1900,115,1.816768,40.0,0,0,0
18507,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,110,67,8900,3100,36,2.169696,68.0,0,0,1
18508,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,176,32,14600,5400,142,0.208669,91.0,0,0,0


In [5]:
from xgboost import XGBRegressor
model = XGBRegressor(n_estimators=2500, learning_rate = 0.023, early_stopping_rounds=10)
model.fit(preprocess(data_train, True),target_train,eval_set=[(preprocess(data_valid), target_valid)], verbose=True)

[0]	validation_0-rmse:706653.83624
[1]	validation_0-rmse:691874.27397
[2]	validation_0-rmse:677536.08971
[3]	validation_0-rmse:663506.93642
[4]	validation_0-rmse:649894.67614
[5]	validation_0-rmse:636553.53504
[6]	validation_0-rmse:623548.44368
[7]	validation_0-rmse:610901.18541
[8]	validation_0-rmse:598518.36393
[9]	validation_0-rmse:586519.00935
[10]	validation_0-rmse:574752.07533
[11]	validation_0-rmse:563240.12047
[12]	validation_0-rmse:552069.35185
[13]	validation_0-rmse:541109.93733
[14]	validation_0-rmse:530496.80611
[15]	validation_0-rmse:520138.19856
[16]	validation_0-rmse:510051.43498
[17]	validation_0-rmse:500221.96098
[18]	validation_0-rmse:490637.55545
[19]	validation_0-rmse:481353.85665
[20]	validation_0-rmse:472290.73060
[21]	validation_0-rmse:463513.62527
[22]	validation_0-rmse:454966.80385
[23]	validation_0-rmse:446622.84478
[24]	validation_0-rmse:438540.71577
[25]	validation_0-rmse:430653.96768
[26]	validation_0-rmse:422863.38600
[27]	validation_0-rmse:415409.02077
[2

XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
             early_stopping_rounds=10, enable_categorical=False,
             eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.023, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=6, max_leaves=0, min_child_weight=1,
             missing=nan, monotone_constraints='()', n_estimators=2500,
             n_jobs=0, num_parallel_tree=1, predictor='auto', random_state=0,
             reg_alpha=0, reg_lambda=1, ...)

In [6]:
pred_valid = model.predict(preprocess(data_valid))
pred_train = model.predict(preprocess(data_train))
print("train pred :",np.mean((target_train-pred_train)**2)**0.5)
#123976/153489(600,0.03)->93291/142582(2000,0.022)->91343/141794(2062,0.023)
print("valid pred :",np.mean((target_valid-pred_valid)**2)**0.5)

train pred : 89663.98525922703
valid pred : 143638.89526227207


In [7]:
model = XGBRegressor(n_estimators=2062, learning_rate = 0.023)
model.fit(preprocess(data, True),target)

XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.023, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=6, max_leaves=0, min_child_weight=1,
             missing=nan, monotone_constraints='()', n_estimators=2062,
             n_jobs=0, num_parallel_tree=1, predictor='auto', random_state=0,
             reg_alpha=0, reg_lambda=1, ...)

In [8]:
pred = model.predict(preprocess(data))
print("train pred :",np.mean((target-pred)**2)**0.5)

train pred : 95273.08510956005


In [9]:
test_data = pd.read_csv('../input/regression220718/test.csv')
test_id = test_data.ID

test_data.drop(['ID','NEAREST_SCH','NEAREST_STN','LONGITUDE','LATITUDE','POSTCODE','ADDRESS'], axis=1, inplace=True)

test_pred = model.predict(preprocess(test_data))

output = pd.DataFrame({'ID': test_id,
                       'PRICE': test_pred})
output.to_csv('submission.csv', index=False)